In [1]:
import datetime
import os
from mdagent import MDAgent
import matplotlib.pyplot as plt

In [2]:
#todays date and time
now = datetime.datetime.now()
date = now.strftime("%Y-%m-%d")
print("date and time:",date)
time = now.strftime("%H:%M:%S")
print("time:",time)

date and time: 2024-07-16
time: 15:20:27


In [3]:
prompt1 = "Simulate pdb  1MBN at two different temperatures: 300K, 400K for 1ns seconds each. Plot RMSD of both over time, and compare the final secondary structures at the end of the simulations."
llm_var = "gpt-3.5-turbo-0125"
tools = "all"
agent = MDAgent(agent_type="Structured", model=llm_var, top_k_tools=tools)

In [4]:
print("LLM: ",agent.llm.model_name,"\nTemperature: ",agent.llm.temperature)

LLM:  gpt-3.5-turbo-0125 
Temperature:  0.1


In [5]:
agent.run(prompt1)

Thought: To simulate the protein 1MBN at different temperatures and compare the RMSD and final secondary structures, I need to set up and run short simulations at 300K and 400K, record the RMSD over time, and analyze the final secondary structures.

Action: SetUpandRunFunction
Action Input: 
{
  "pdb_id": "1MBN",
  "forcefield_files": ["amber14/protein.ff14SB.xml", "amber14/tip3p.xml"],
  "save": true,
  "system_params": {
    "nonbondedMethod": "NoCutoff",
    "constraints": "HBonds",
    "rigidWater": true
  },
  "integrator_params": {
    "integrator_type": "LangevinMiddle",
    "Temperature": "300 * kelvin",
    "Friction": "1.0 / picoseconds",
    "Timestep": "0.002 * picoseconds"
  },
  "simulation_params": {
    "Ensemble": "NVT",
    "Number of Steps": 500000,
    "record_interval_steps": 1000,
    "record_params": ["step", "rmsd", "temperature"]
  }
}The agent's thought was to simulate the protein 1MBN at different temperatures (300K and 400K) and compare the RMSD and final se

('Thought: To simulate the protein 1MBN at different temperatures and compare the RMSD and final secondary structures, I need to set up and run short simulations at 300K and 400K, record the RMSD over time, and analyze the final secondary structures.\n\nAction: SetUpandRunFunction\nAction Input: \n{\n  "pdb_id": "1MBN",\n  "forcefield_files": ["amber14/protein.ff14SB.xml", "amber14/tip3p.xml"],\n  "save": true,\n  "system_params": {\n    "nonbondedMethod": "NoCutoff",\n    "constraints": "HBonds",\n    "rigidWater": true\n  },\n  "integrator_params": {\n    "integrator_type": "LangevinMiddle",\n    "Temperature": "300 * kelvin",\n    "Friction": "1.0 / picoseconds",\n    "Timestep": "0.002 * picoseconds"\n  },\n  "simulation_params": {\n    "Ensemble": "NVT",\n    "Number of Steps": 500000,\n    "record_interval_steps": 1000,\n    "record_params": ["step", "rmsd", "temperature"]\n  }\n}',
 '541886F0')

In [6]:
#print final date and time
now = datetime.datetime.now()
date = now.strftime("%Y-%m-%d")
print("date and time:",date)
time = now.strftime("%H:%M:%S")
print("time:",time)

date and time: 2024-07-16
time: 15:20:34


In [7]:
registry = agent.path_registry
print(registry.list_path_names_and_descriptions())

No names found. The JSON file is empty or does not contain name mappings.


In [8]:
paths_and_descriptions = registry.list_path_names_and_descriptions()
print("\n".join(paths_and_descriptions.split(",")))

No names found. The JSON file is empty or does not contain name mappings.


In [9]:
#plotting rmsd of both simulations
from IPython.display import Image
rmsd1ID = 'fig0_200719'
#rmsd2ID = 'fig0_165231'
path1 = registry.get_mapped_path(rmsd1ID)
#path2 = registry.get_mapped_path(rmsd2ID)

Image(filename=path1)




ValueError: Cannot embed the '' image format

In [ ]:
import mdtraj as md
import numpy as np

traj_path_1 = registry.get_mapped_path("rec2_192450")
top_path_1 = registry.get_mapped_path("top_sim0_192450")
traj_1 = md.load(traj_path_1, top=top_path_1)

traj_path_2 = registry.get_mapped_path("rec2_194650")
top_path_2 = registry.get_mapped_path("top_sim0_194650")
traj_2 = md.load(traj_path_2, top=top_path_2)


# Compute the secondary structure of the trajectory
dssp_300 = md.compute_dssp(traj_1, simplified=True)
dssp_400 = md.compute_dssp(traj_2, simplified=True)
print("number of residues: ", traj_1.n_residues)
compl_list = ['E', 'H', 'C']
print("Number of sheets 300K: ",len([i for i in dssp_300.flatten() if i == 'E' ])/traj_1.n_frames)
print("Number of helices 300K: ",len([i for i in dssp_300.flatten() if i == 'H'])/traj_1.n_frames)
print("Number of coils 300K: ",len([i for i in dssp_300.flatten() if i == 'C'])/traj_1.n_frames)
print("Number of residues not in E, H, or C: ", len([i for i in dssp_300.flatten() if i not in compl_list])/traj_1.n_frames)

print("number of residues: ", traj_2.n_residues)
print("Number of sheets 400K: ",len([i for i in dssp_400.flatten() if i == 'E'])/traj_2.n_frames)
print("Number of helices 400K: ",len([i for i in dssp_400.flatten() if i == 'H'])/traj_2.n_frames)
print("Number of coils: 400k",len([i for i in dssp_400.flatten() if i == 'C'])/traj_2.n_frames)
print("Number of residues not in E, H, or C: ", len([i for i in dssp_400.flatten() if i not in compl_list])/traj_1.n_frames)

print("Agent response: ", "At 300K, there were 505,414 helices, 0 strands, and 261,116 coils. At 400K, there were 505,397 helices, 712 strands, and 260,421 coils.")

number of residues:  153
Number of sheets 300K:  0.0
Number of helices 300K:  100.87664670658683
Number of coils 300K:  52.123353293413174
Number of residues not in E, H, or C:  0.0
number of residues:  153
Number of sheets 400K:  0.14211576846307386
Number of helices 400K:  100.87644710578843
Number of coils: 400k 51.981437125748506
Number of residues not in E, H, or C:  0.0
Agent response:  At 300K, there were 505,414 helices, 0 strands, and 261,116 coils. At 400K, there were 505,397 helices, 712 strands, and 260,421 coils.


# Experiment Result:
### Completed without Exception or TimeOut Errors ✅
### Attempted all necessary steps ❌
### Logic make sense ❌
### Correct Answer  ❌